# Phase 2a: YOLOv8s Baseline Model Training on Kaggle

This notebook executes an automated end-to-end training and evaluation workflow for the YOLOv8s baseline model on the NEU-DET surface defect dataset.

### Pipeline:
1. **Repository Setup**: Clones `https://github.com/AaruneshAP/defect-inspection.git` on branch `main` and installs required packages.
2. **Dataset Discovery**: Dynamically discovers the NEU-DET archive under `/kaggle/input`. If absent, raises a clear error instructing to attach the dataset.
3. **YOLO Formatting**: Converts Pascal VOC annotations to normalized YOLO format (`data/yolo/`) with a deterministic 85/15 stratified train/val split (`seed=42`).
4. **Baseline Model Training**: Trains `yolov8s.pt` with early-stopping patience using `configs/baseline.yaml` (`imgsz=640`, `batch=16`).
5. **Strict Validation Evaluation**: Evaluates detection performance strictly on the `val` split (test split is completely untouched).
6. **Artifact Export**: Saves `results.json` (mAP50, mAP50-95, precision, recall, per-class AP), confusion matrix, PR curve, and training curves.


In [ ]:
import os
import subprocess
from pathlib import Path

# --- CONFIGURATION VARIABLES ---
REPO_URL = "https://github.com/AaruneshAP/defect-inspection.git"
BRANCH_NAME = "main"
REPO_NAME = "defect-inspection"

working_dir = Path("/kaggle/working")
if working_dir.exists():
    target_repo = working_dir / REPO_NAME
    if not target_repo.exists():
        subprocess.run(["git", "clone", "-b", BRANCH_NAME, REPO_URL], check=True, cwd=str(working_dir))
    os.chdir(target_repo)
else:
    print("Running in local environment; repository root confirmed.")

subprocess.run(["pip", "install", "-q", "ultralytics", "pyyaml"], check=True)


## 1. Discover NEU-DET Dataset under `/kaggle/input`

We search for the NEU-DET dataset directory under `/kaggle/input`. If not found, a clear FileNotFoundError is raised directing the user to attach the dataset.


In [ ]:
from pathlib import Path

kaggle_root = Path("/kaggle/input")
if not kaggle_root.exists():
    raise FileNotFoundError("Attach the NEU Surface Defect Database under Add Input")

candidates = list(kaggle_root.rglob("*NEU*"))
raw_data_dir = None
for cand in candidates:
    if (cand / "train").exists() or (cand / "NEU-DET").exists() or list(cand.rglob("*.xml")):
        raw_data_dir = cand
        break

if raw_data_dir is None:
    raise FileNotFoundError("Attach the NEU Surface Defect Database under Add Input")

print(f"Dataset root identified at: {raw_data_dir.resolve()}")


## 2. Ingest and Convert Dataset to YOLO Format

Runs `convert_dataset_to_yolo` to generate normalized YOLO bounding boxes and the 85/15 stratified train/val split.


In [ ]:
from src.data import convert_dataset_to_yolo

conversion_summary = convert_dataset_to_yolo(
    data_dir=raw_data_dir,
    output_dir="data/yolo",
    val_ratio=0.15,
    seed=42,
)

print("YOLO dataset conversion completed successfully:")
print(f"  Train images: {conversion_summary['counts']['train']}")
print(f"  Val images:   {conversion_summary['counts']['val']}")
print(f"  Test images:  {conversion_summary['counts']['test']} (reserved)")


## 3. Train YOLOv8s Baseline Model

Loads `configs/baseline.yaml` and executes training with early stopping.

### Image Resolution:
The baseline starts with `imgsz=640` to maximize FPN feature resolution for small defect structures. A comparison variant at `imgsz=320` is provided in `configs/baseline_320.yaml`. Both are exact multiples of stride 32.


In [ ]:
import yaml
import torch
from src.train import load_config, run_training

config = load_config("configs/baseline.yaml")
print("Training Configuration:")
print(yaml.dump(config, default_flow_style=False))

device = 0 if torch.cuda.is_available() else "cpu"
print(f"Executing training on device: {device}")

train_summary = run_training(
    config_path="configs/baseline.yaml",
    device=device,
)

print()
print("Baseline training completed successfully!")
print("Best checkpoint saved at:", train_summary["best_checkpoint"])


## 4. Strict Validation Evaluation (Val Split Only)

Evaluates the best checkpoint strictly on the `val` split. The test split is never evaluated here to protect against data snooping.


In [ ]:
from src.train import evaluate

best_checkpoint = train_summary["best_checkpoint"]
val_metrics = evaluate(
    model=best_checkpoint,
    data_yaml="data/yolo/data.yaml",
    split="val",
    imgsz=config["imgsz"],
    device=device,
)

print("=" * 60)
print("       YOLOv8s BASELINE VALIDATION PERFORMANCE")
print("=" * 60)
print(f"Precision: {val_metrics['precision']:.4f}")
print(f"Recall:    {val_metrics['recall']:.4f}")
print(f"mAP@50:    {val_metrics['mAP50']:.4f}")
print(f"mAP@50-95: {val_metrics['mAP50-95']:.4f}")
print("-" * 60)
print(f"{'Class':<18} | {'AP@50':<10} | {'AP@50-95':<10}")
print("-" * 60)
for cls_name, aps in val_metrics["per_class_ap"].items():
    print(f"{cls_name:<18} | {aps['ap50']:<10.4f} | {aps['ap50-95']:<10.4f}")
print("=" * 60)


## 5. Artifact Inspection & Export

Saves `results.json` and renders the confusion matrix, PR curve, and loss curves.


In [ ]:
from pathlib import Path
from IPython.display import Image, display

run_dir = Path(train_summary["run_dir"])

results_file = Path(train_summary["results_json"])
print(f"--- Summary results saved to {results_file} ---")
with open(results_file, "r", encoding="utf-8") as f:
    print(f.read())

for cm_name in ["confusion_matrix_normalized.png", "confusion_matrix.png"]:
    cm_path = run_dir / cm_name
    if cm_path.exists():
        print()
        print(f"Confusion Matrix ({cm_name}):")
        display(Image(filename=str(cm_path)))
        break

pr_path = run_dir / "PR_curve.png"
if pr_path.exists():
    print()
    print("Precision-Recall Curve:")
    display(Image(filename=str(pr_path)))

results_plot = run_dir / "results.png"
if results_plot.exists():
    print()
    print("Training and Validation Curves:")
    display(Image(filename=str(results_plot)))
